# Hierarchical Clustering

## Overview

Hierarchical clustering builds a tree (dendrogram) of nested clusters without requiring k to be specified in advance. Two approaches:

- **Agglomerative (bottom-up):** each point starts as its own cluster; clusters are merged iteratively by linkage criterion
- **Divisive (top-down):** start with one cluster and split recursively (less common)

**Linkage methods:**

| Method | Merges by | Sensitive to |
|---|---|---|
| `ward` | Minimises within-cluster variance | Outliers (but robust overall) |
| `complete` | Maximum distance between clusters | Outliers |
| `average` | Mean pairwise distance | Shape |
| `single` | Minimum distance (chaining) | Can produce elongated chains |

`ward` linkage is the default choice for most ecological and tabular data.

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from sklearn.cluster import AgglomerativeClustering
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

rng = np.random.default_rng(42)
centers = [(50,6.5),(200,7.2),(350,7.8),(150,6.9)]
X_list = []
for cx, cy in centers:
    n = rng.integers(40, 55)
    X_list.append(np.column_stack([
        rng.normal(cx,30,n), rng.normal(cy,0.25,n),
        rng.gamma(2,1.5,n),  rng.normal(7,0.8,n)
    ]))
X = np.vstack(X_list)
feat_names = ["elevation","ph","nitrate","phosphorus"]
scaler = StandardScaler()
X_sc = scaler.fit_transform(X)
print(f"Dataset: {X.shape}")

---
## Dendrogram

In [ ]:
Z = linkage(X_sc, method="ward")
fig, ax = plt.subplots(figsize=(12,5))
dendrogram(Z, ax=ax, truncate_mode="lastp", p=20,
           leaf_rotation=45, leaf_font_size=9,
           color_threshold=0.7*max(Z[:,2]))
ax.set_title("Ward Linkage Dendrogram (truncated to last 20 merges)")
ax.set_xlabel("Sample index or cluster size")
ax.set_ylabel("Distance")
plt.tight_layout(); plt.show()
print("Cut the dendrogram where the longest vertical lines appear -> suggests k")

---
## Cutting the Dendrogram

In [ ]:
# Extract flat clusters by cutting at different heights
for k in [3, 4, 5]:
    labels = fcluster(Z, k, criterion="maxclust") - 1  # 0-indexed
    sil = silhouette_score(X_sc, labels)
    print(f"k={k}: silhouette={sil:.3f}, sizes={np.bincount(labels).tolist()}")

best_labels = fcluster(Z, 4, criterion="maxclust") - 1
df = pd.DataFrame(X, columns=feat_names)
df["cluster"] = best_labels
print("\nCluster means:")
print(df.groupby("cluster")[feat_names].mean().round(2))

---
## Linkage Method Comparison

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12,8))
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_sc)
for ax, method in zip(axes.flatten(), ["ward","complete","average","single"]):
    Z_m = linkage(X_sc, method=method)
    labels_m = fcluster(Z_m, 4, criterion="maxclust") - 1
    sil = silhouette_score(X_sc, labels_m)
    for k in range(4):
        mask = labels_m == k
        ax.scatter(X_pca[mask,0], X_pca[mask,1], s=20, alpha=0.7)
    ax.set_title(f"{method} linkage  (sil={sil:.3f})")
    ax.set_xlabel("PC1"); ax.set_ylabel("PC2")
plt.suptitle("Linkage Method Comparison in PCA Space", y=1.01)
plt.tight_layout(); plt.show()

---
## sklearn AgglomerativeClustering

In [ ]:
# sklearn interface: useful for connectivity constraints and large data
agg = AgglomerativeClustering(n_clusters=4, linkage="ward")
labels_agg = agg.fit_predict(X_sc)
print(f"AgglomerativeClustering silhouette: {silhouette_score(X_sc, labels_agg):.3f}")
print(f"Cluster sizes: {np.bincount(labels_agg).tolist()}")
# Connectivity constraint: only merge spatially adjacent clusters
from sklearn.neighbors import kneighbors_graph
conn = kneighbors_graph(X_sc, n_neighbors=10, include_self=False)
agg_conn = AgglomerativeClustering(n_clusters=4, linkage="ward", connectivity=conn)
labels_conn = agg_conn.fit_predict(X_sc)
print(f"With connectivity constraint: {silhouette_score(X_sc, labels_conn):.3f}")

---

## Common Pitfalls

**1. Using single linkage for general clustering**  
Single linkage merges clusters based on the minimum pairwise distance, which produces chain-like clusters that connect distant points through intermediate ones. Use Ward or average linkage for compact, natural clusters.

**2. Not standardising before computing linkage**  
Linkage distances are Euclidean by default. Features on different scales dominate the distance. Always standardise before calling `linkage()`.

**3. Reading the dendrogram without checking silhouette scores**  
The dendrogram suggests k visually, but the "long vertical line" heuristic is subjective. Confirm the choice by computing silhouette scores for candidate k values after cutting.

**4. Applying hierarchical clustering to very large datasets**  
The linkage matrix computation is O(n^2) in memory. For n > 10,000 use mini-batch k-means or HDBSCAN instead. For moderate n (1,000–10,000), `scipy.cluster.hierarchy` is efficient enough.

**5. Confusing `fcluster` 1-based labels with sklearn 0-based labels**  
`scipy.fcluster` returns labels starting from 1; sklearn returns labels starting from 0. Mixing them without adjustment causes off-by-one errors in cluster profiling and silhouette computation.
---
*python_methods_library - Samantha McGarrigle*